# Chapter 0 — Prerequisites · Practice

**Goal:** prove your environment works end to end: the right Python, every package importable, both API keys loaded (never printed), one tiny call to Claude and one to OpenAI, and the habit of counting tokens and estimating cost before you spend money.

- Theory: [README.md](README.md)
- **Estimated time:** 45–60 minutes
- **Estimated API cost:** $0 by default. With `RUN_LIVE = True`, about $0.01 or less in total with the default models (`claude-sonnet-5-5` at `effort: low`, and `gpt-6-luna`); token counting is free on Claude
- **Prerequisites:** README steps 1–6 done (venv, `pip install -r requirements.txt`, `.env` created); run from the repo venv kernel

Everything runs **offline** unless you set `RUN_LIVE = True` in the setup cell. Offline cells use a fake HTTP transport, so you see real SDK objects, real request bodies and real SDK exceptions without a key or credit.

In [ ]:
import asyncio
import importlib.metadata as metadata
import json
import os
import platform
import random
import shutil
import subprocess
import sys
import time
from pathlib import Path

import anthropic
import httpx2
import openai
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
CLAUDE_MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-5-5")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")

# Paid calls are opt-in. Set to True only after section 0.2 shows both keys as "set".
RUN_LIVE = False

print("Claude model:", CLAUDE_MODEL, "| OpenAI model:", OPENAI_MODEL, "| RUN_LIVE:", RUN_LIVE)

## 0.1 Python version and packages

The repo needs Python 3.11+ and the packages in `requirements.txt`. This cell reads the lower bounds straight from that file, so it stays correct when the file changes. It also reports which **optional extras** from `requirements-extras.txt` are installed (missing extras are fine). [Theory](README.md#step-1--install-python-311) · [Theory](README.md#step-3--install-the-shared-dependencies)

In [ ]:
def find_repo_file(name: str) -> Path | None:
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / name).exists():
            return folder / name
    return None


def version_tuple(v: str) -> tuple[int, ...]:
    parts = []
    for piece in v.split("."):
        digits = "".join(ch for ch in piece if ch.isdigit())
        if not digits:
            break
        parts.append(int(digits))
    return tuple(parts)


def read_requirements(path: Path) -> dict[str, str]:
    reqs = {}
    for line in path.read_text().splitlines():
        line = line.split("#", 1)[0].strip()
        if ">=" in line:
            name, minimum = (s.strip() for s in line.split(">=", 1))
            reqs[name.split("[", 1)[0]] = minimum.split(",", 1)[0]
    return reqs


IMPORT_NAMES = {"claude-agent-sdk": "claude_agent_sdk", "openai-agents": "agents",
                "python-dotenv": "dotenv", "pyyaml": "yaml"}

py_ok = sys.version_info >= (3, 11)
print(f"Python {platform.python_version()} -> {'OK' if py_ok else 'TOO OLD: install 3.11+'}")
print(f"Interpreter: {sys.executable}")
print("Inside a virtual environment:", sys.prefix != sys.base_prefix)

req_file = find_repo_file("requirements.txt")
problems = 0
if req_file is None:
    print("requirements.txt not found: start Jupyter from the repo folder")
    problems += 1
else:
    print(f"\n{'package':<18} {'installed':<12} {'minimum':<10} status")
    for dist, minimum in read_requirements(req_file).items():
        try:
            installed = metadata.version(dist)
        except metadata.PackageNotFoundError:
            print(f"{dist:<18} {'-':<12} {minimum:<10} MISSING (pip install -r requirements.txt)")
            problems += 1
            continue
        ok = version_tuple(installed) >= version_tuple(minimum)
        problems += not ok
        print(f"{dist:<18} {installed:<12} {minimum:<10} {'OK' if ok else 'TOO OLD (pip install -U ' + dist + ')'}")

for dist in ("anthropic", "openai", "claude-agent-sdk", "mcp", "openai-agents", "pyyaml", "httpx", "httpx2"):
    try:
        __import__(IMPORT_NAMES.get(dist, dist))
    except Exception as e:  # noqa: BLE001
        print(f"import {IMPORT_NAMES.get(dist, dist)} failed: {type(e).__name__}: {e}")
        problems += 1
print("\nCore environment:", "OK" if problems == 0 and py_ok else f"{problems} problem(s), fix before going on")

In [ ]:
# Optional extras (requirements-extras.txt): only needed by some later chapters.
EXTRAS = {
    "fastembed": "local embeddings (applied-ai-architect 06-07, capstone 05)",
    "chromadb": "local vector store (applied-ai-architect 06-07, capstone 05)",
    "bm25s": "lexical BM25 search for hybrid retrieval (applied-ai-architect 06)",
    "pydantic-ai-slim": "agent framework (anthropic-claude 21, openai-codex 21)",
    "openai-codex": "Codex SDK for Python (openai-codex 05; commented out in the extras file)",
}
for dist, why in EXTRAS.items():
    try:
        status = f"installed {metadata.version(dist)}"
    except metadata.PackageNotFoundError:
        status = "not installed (fine for now)"
    print(f"{dist:<18} {status:<30} {why}")
node = shutil.which("node")
print(f"{'node':<18} {'found' if node else 'not found (fine for now)':<30} Promptfoo eval CLI via npx (openai-codex 06, applied-ai-architect 09)")

> **Observe:** the interpreter path should point inside the repo's `.venv`. If it points to a system Python, the notebook is on the wrong kernel (README Step 8).

> **Try it:** in a terminal, run `pip index versions anthropic` to see whether a newer SDK exists. Lower bounds in `requirements.txt` are minimums, not pins.

## 0.2 API keys and `.env` hygiene

The SDKs read `ANTHROPIC_API_KEY` and `OPENAI_API_KEY` from the environment. This cell checks they are present and look like the right kind of key **without ever printing them**, and checks that Git ignores `.env`. [Theory](README.md#step-6--create-your-env) · [Theory](README.md#35-security-and-secrets)

In [ ]:
KEY_PREFIXES = {"ANTHROPIC_API_KEY": "sk-ant-", "OPENAI_API_KEY": "sk-"}


def key_status(var: str) -> str:
    value = os.getenv(var, "")
    if not value:
        return "MISSING"
    shape = "format looks right" if value.startswith(KEY_PREFIXES[var]) else "unexpected format, check for a copy/paste error"
    return f"set ({shape})"        # never print the value, its length or its last characters


for var in KEY_PREFIXES:
    print(f"{var:<18}: {key_status(var)}")
HAVE_CLAUDE_KEY = bool(os.getenv("ANTHROPIC_API_KEY"))
HAVE_OPENAI_KEY = bool(os.getenv("OPENAI_API_KEY"))

env_file = find_dotenv()
print("\n.env file:", env_file or "not found (copy .env.example to .env)")
try:
    root = Path(env_file).parent if env_file else (find_repo_file("requirements.txt") or Path("requirements.txt")).parent
    ignored = subprocess.run(["git", "check-ignore", "-q", ".env"], cwd=root, capture_output=True).returncode == 0
    print(".env ignored by git:", "yes, safe" if ignored else "NO: add .env to .gitignore before committing")
except FileNotFoundError:
    print("git not installed: cannot check .gitignore")

In [ ]:
SECRET_HEADERS = {"x-api-key", "authorization"}


def redact_headers(headers) -> dict[str, str]:
    """Copy of request headers that is safe to print or log."""
    return {k: ("<redacted>" if k.lower() in SECRET_HEADERS else v) for k, v in dict(headers).items()}


print(redact_headers({"x-api-key": "sk-ant-not-a-real-key", "anthropic-version": "2023-06-01"}))

> **Observe:** `key_status` never reveals any part of the key, not even its length or last four characters. Treat notebook output as public: it gets committed and screenshotted.

> **Try it:** temporarily rename `.env` and restart the kernel. Section 0.2 should report both keys as `MISSING`, and every later cell should still run offline.

## 0.3 Next-token generation: a toy model

A model produces text one token at a time: compute a probability for every possible next token, sample one, append it, repeat. This toy "model" is a hand-written probability table, but the loop is the real one. [Theory](README.md#29-next-token-generation) · [Theory](README.md#23-temperature-and-sampling)

In [ ]:
# Next-token probabilities given the previous token (a real model conditions on the whole context).
TOY_MODEL = {
    "<start>": {"The": 0.6, "Your": 0.4},
    "The": {"refund": 0.5, "order": 0.3, "invoice": 0.2},
    "Your": {"refund": 0.4, "order": 0.6},
    "refund": {"was": 0.7, "is": 0.3},
    "order": {"was": 0.5, "is": 0.5},
    "invoice": {"was": 0.6, "is": 0.4},
    "was": {"approved.": 0.6, "rejected.": 0.3, "<end>": 0.1},
    "is": {"pending.": 0.8, "<end>": 0.2},
}


def generate(rng: random.Random, max_tokens: int = 10) -> tuple[str, str]:
    tokens, prev = [], "<start>"
    for _ in range(max_tokens):
        options = TOY_MODEL.get(prev, {"<end>": 1.0})
        nxt = rng.choices(list(options), weights=list(options.values()))[0]   # sample, not argmax
        if nxt == "<end>":
            return " ".join(tokens), "end_turn"
        tokens.append(nxt)
        prev = nxt if not nxt.endswith(".") else "<done>"
        if prev == "<done>":
            return " ".join(tokens), "end_turn"
    return " ".join(tokens), "max_tokens"


rng = random.Random(7)
for _ in range(6):
    print(generate(rng))
print("\nWith max_tokens=2:", generate(random.Random(1), max_tokens=2))

In [ ]:
# Testing output that varies: assert properties and measure a pass rate, not one exact string.
def passes(text: str) -> bool:
    words = text.split()
    return len(words) >= 3 and words[-1].endswith(".")   # a complete sentence


rng = random.Random(0)
runs = [generate(rng)[0] for _ in range(200)]
print("distinct outputs:", len(set(runs)))
print(f"pass rate of the property test: {sum(map(passes, runs)) / len(runs):.0%}")

> **Observe:** the same "prompt" gives different sentences, and some runs stop early. An exact-match test (`assert text == "The refund was approved."`) would be flaky; a property test with a pass rate tells you how often the system is acceptable.

> **Try it:** change the weights of `"was"` so `"<end>"` is 0.5 and re-run. The pass rate drops: that is what a regression looks like in an eval.

## 0.4 Anatomy of an API call (offline)

Under each SDK is one HTTPS `POST` with a JSON body. Here both SDKs talk to a **fake transport** (`httpx2.MockTransport`) that records each request and returns a canned response, so you can see the wire format and the typed response objects without a key. [Theory](README.md#12-http--rest-and-json) · [Theory](README.md#25-system-prompts-vs-user-prompts) · [Theory](README.md#24-thinking-models-and-effort)

In [ ]:
WIRE_LOG: list[httpx2.Request] = []

CLAUDE_REPLY = {
    "id": "msg_mock_01", "type": "message", "role": "assistant", "model": "claude-sonnet-5-5",
    "content": [
        {"type": "thinking", "thinking": "The user wants a fixed phrase.", "signature": "mock-signature"},
        {"type": "text", "text": "hello from Claude"},
    ],
    "stop_reason": "end_turn", "stop_sequence": None,
    "usage": {"input_tokens": 21, "output_tokens": 14, "cache_read_input_tokens": 0, "cache_creation_input_tokens": 0},
}
OPENAI_REPLY = {
    "id": "resp_mock_01", "object": "response", "created_at": 1790000000, "status": "completed",
    "model": "gpt-6-luna",
    "output": [
        {"type": "reasoning", "id": "rs_mock_01", "summary": []},
        {"type": "message", "id": "msg_mock_02", "role": "assistant", "status": "completed",
         "content": [{"type": "output_text", "text": "hello from OpenAI", "annotations": []}]},
    ],
    "usage": {"input_tokens": 18, "input_tokens_details": {"cached_tokens": 0},
              "output_tokens": 22, "output_tokens_details": {"reasoning_tokens": 12}, "total_tokens": 40},
}


def fake_api(request: httpx2.Request) -> httpx2.Response:
    """Pretend to be api.anthropic.com and api.openai.com."""
    WIRE_LOG.append(request)
    path, headers = request.url.path, request.headers
    if headers.get("x-api-key") == "bad-key" or headers.get("authorization") == "Bearer bad-key":
        body = {"type": "error", "error": {"type": "authentication_error", "message": "invalid x-api-key"},
                "request_id": "req_mock_401"}
        return httpx2.Response(401, json=body, headers={"request-id": "req_mock_401"})
    if path == "/v1/messages":
        return httpx2.Response(200, json=CLAUDE_REPLY, headers={"request-id": "req_mock_claude"})
    if path == "/v1/messages/count_tokens":
        return httpx2.Response(200, json={"input_tokens": 21})
    if path == "/v1/responses":
        return httpx2.Response(200, json=OPENAI_REPLY, headers={"x-request-id": "req_mock_openai"})
    if path == "/v1/responses/input_tokens":
        return httpx2.Response(200, json={"object": "response.input_tokens", "input_tokens": 18})
    return httpx2.Response(404, json={"type": "error", "error": {"type": "not_found_error", "message": path}})


def mock_clients(api_key: str = "sk-ant-offline-demo", max_retries: int = 2, handler=fake_api):
    """Real SDK clients whose HTTP traffic goes to `handler` instead of the internet."""
    claude = anthropic.Anthropic(api_key=api_key, max_retries=max_retries,
                                 http_client=httpx2.Client(transport=httpx2.MockTransport(handler)))
    oai = openai.OpenAI(api_key=api_key.replace("sk-ant-", "sk-"), max_retries=max_retries,
                        http_client=httpx2.Client(transport=httpx2.MockTransport(handler)))
    return claude, oai


mock_claude, mock_oai = mock_clients()

In [ ]:
# Claude: system prompt at the top level, user turn in `messages`, effort in `output_config`.
mock_msg = mock_claude.messages.create(
    model=CLAUDE_MODEL,
    max_tokens=1024,
    system="You are a terse assistant.",
    messages=[{"role": "user", "content": "Reply with exactly: hello from Claude"}],
    output_config={"effort": "low"},
)
req = WIRE_LOG[-1]
print(req.method, req.url)
print(json.dumps(redact_headers({k: v for k, v in req.headers.items() if not k.startswith("x-stainless")}), indent=1))
print(json.dumps(json.loads(req.content), indent=1))

print("\nblock types:", [b.type for b in mock_msg.content])
text = "".join(b.text for b in mock_msg.content if b.type == "text")   # never assume content[0] is the answer
print("text:", text, "| stop_reason:", mock_msg.stop_reason, "| request id:", mock_msg._request_id)
print("usage:", mock_msg.usage.input_tokens, "in /", mock_msg.usage.output_tokens, "out")

In [ ]:
# OpenAI: instructions + input, reasoning effort, and store=False for a throwaway call.
mock_resp = mock_oai.responses.create(
    model=OPENAI_MODEL,
    instructions="You are a terse assistant.",
    input="Reply with exactly: hello from OpenAI",
    reasoning={"effort": "low"},
    max_output_tokens=512,
    store=False,
)
req = WIRE_LOG[-1]
print(req.method, req.url)
print(json.dumps(json.loads(req.content), indent=1))
print("\noutput item types:", [item.type for item in mock_resp.output])
print("output_text:", mock_resp.output_text, "| status:", mock_resp.status, "| request id:", mock_resp._request_id)
u = mock_resp.usage
print("usage:", u.input_tokens, "in /", u.output_tokens, "out, of which reasoning:", u.output_tokens_details.reasoning_tokens)

> **Observe:** the Claude response starts with a `thinking` block and the OpenAI response with a `reasoning` item. Code that reads `content[0].text` or `output[0]` breaks on thinking models; select by type (or use `output_text`). On OpenAI, reasoning tokens are part of `output_tokens` and billed as output; on Claude, thinking tokens are also billed as output.

> **Try it:** remove `output_config` from the Claude call and look at the request body again. The SDK only sends what you pass.

## 0.5 Failure modes and SDK retries (offline)

Failures are normal. The SDKs raise typed exceptions and retry transient ones on their own (2 retries by default, honoring `retry-after`). These fake servers fail on purpose. [Theory](README.md#33-retries-and-backoff) · [Theory](README.md#12-http--rest-and-json)

In [ ]:
# 1) A wrong key: 401 -> AuthenticationError. Never retried; fix the credentials.
bad_claude, bad_oai = mock_clients(api_key="bad-key")
for name, call in [
    ("Claude", lambda: bad_claude.messages.create(model=CLAUDE_MODEL, max_tokens=16,
                                                  messages=[{"role": "user", "content": "hi"}])),
    ("OpenAI", lambda: bad_oai.responses.create(model=OPENAI_MODEL, input="hi")),
]:
    try:
        call()
    except (anthropic.AuthenticationError, openai.AuthenticationError) as e:
        print(f"{name}: {type(e).__module__}.{type(e).__name__}, status {e.status_code}")

In [ ]:
def flaky(fail_status: int, fail_times: int, headers: dict | None = None, error_code: str | None = None):
    """A fake server that fails `fail_times` times, then behaves like fake_api."""
    state = {"calls": 0}

    def handler(request: httpx2.Request) -> httpx2.Response:
        state["calls"] += 1
        retry_count = request.headers.get("x-stainless-retry-count")
        print(f"  attempt {state['calls']} (x-stainless-retry-count={retry_count})")
        if state["calls"] <= fail_times:
            err = {"type": "overloaded_error" if fail_status == 529 else "rate_limit_error", "message": "simulated"}
            if error_code:
                err["details"] = {"error_code": error_code}
            return httpx2.Response(fail_status, json={"type": "error", "error": err}, headers=headers or {})
        return fake_api(request)

    return handler, state


# 2) Overloaded (529) once, with a short retry-after: the SDK retries for you and succeeds.
handler, state = flaky(529, fail_times=1, headers={"retry-after-ms": "50"})
claude, _ = mock_clients(handler=handler)
retried = claude.messages.create(model=CLAUDE_MODEL, max_tokens=16, messages=[{"role": "user", "content": "hi"}])
print("succeeded after", state["calls"], "attempts:", retried.content[-1].text)

In [ ]:
# 3) Spend-cap 429: no retry-after header, error_code says the cap is reached. Retrying cannot help.
handler, state = flaky(429, fail_times=99, error_code="enforced_spend_limit_reached")
claude, _ = mock_clients(handler=handler)
t0 = time.perf_counter()
try:
    claude.messages.create(model=CLAUDE_MODEL, max_tokens=16, messages=[{"role": "user", "content": "hi"}])
except anthropic.RateLimitError as e:
    code_ = (e.body or {}).get("error", {}).get("details", {}).get("error_code")
    print(f"gave up after {state['calls']} attempts in {time.perf_counter() - t0:.1f}s; error_code={code_}")
    if code_ == "enforced_spend_limit_reached":
        print("-> not transient: alert a human instead of retrying")

In [ ]:
RETRYABLE = {408, 409, 429, 500, 502, 503, 504, 529}


def should_retry(err: Exception) -> bool:
    """Decide whether YOUR code (a queue worker, a job runner) should try again later."""
    if isinstance(err, (anthropic.APIConnectionError, openai.APIConnectionError)):
        return True                                  # includes timeouts
    status = getattr(err, "status_code", None)
    body = getattr(err, "body", None) or {}
    details = body.get("error", {}).get("details", {}) if isinstance(body, dict) else {}
    if status == 429 and details.get("error_code") == "enforced_spend_limit_reached":
        return False
    return status in RETRYABLE


try:
    bad_claude.messages.create(model=CLAUDE_MODEL, max_tokens=16, messages=[{"role": "user", "content": "hi"}])
except anthropic.APIStatusError as e:
    print("401 ->", "retry" if should_retry(e) else "do not retry")

> **Observe:** the 529 case succeeded on attempt 2 without any retry code from you. The spend-cap case still cost three attempts, because the SDK cannot tell a spend cap from a rate limit; your code has to read `error_code`. Retries also multiply load: cap them.

> **Try it:** create a client with `max_retries=0` in `mock_clients(...)` and re-run case 2. It now raises `anthropic.OverloadedError` (the SDK class for 529) on the first failure.

## 0.6 First live call to Claude and OpenAI (opt-in)

The README smoke test, guarded: it runs only if `RUN_LIVE = True` **and** the key is present. Cost: a fraction of a cent per vendor. [Theory](README.md#step-9--smoke-test-both-apis) · [Theory](README.md#step-7--put-cost-guardrails-in-place-before-the-first-call)

In [ ]:
def claude_effort_kwargs(model: str) -> dict:
    # Haiku 4.5 does not support `effort`; current Sonnet/Opus/Fable models do.
    return {} if model.startswith("claude-haiku") else {"output_config": {"effort": "low"}}


LIVE = {}
if RUN_LIVE and HAVE_CLAUDE_KEY:
    try:
        live_claude = anthropic.Anthropic()
        t0 = time.perf_counter()
        live_msg = live_claude.messages.create(
            model=CLAUDE_MODEL, max_tokens=1024,
            messages=[{"role": "user", "content": "Reply with exactly: hello from Claude"}],
            **claude_effort_kwargs(CLAUDE_MODEL),
        )
        LIVE["claude"] = (live_msg, (time.perf_counter() - t0) * 1000)
        text = "".join(b.text for b in live_msg.content if b.type == "text")
        print(text, "|", live_msg.stop_reason, "|", live_msg.usage.input_tokens, "in /", live_msg.usage.output_tokens, "out |", live_msg._request_id)
    except anthropic.APIStatusError as e:
        print(f"Claude call failed: {type(e).__name__} {e.status_code}. See the README troubleshooting table.")
    except anthropic.APIConnectionError:
        print("Claude call failed: no network connection")
else:
    print("Skipped: set RUN_LIVE = True and put ANTHROPIC_API_KEY in .env to run this cell.")

In [ ]:
if RUN_LIVE and HAVE_OPENAI_KEY:
    try:
        live_oai = openai.OpenAI()
        t0 = time.perf_counter()
        live_resp = live_oai.responses.create(
            model=OPENAI_MODEL,
            input="Reply with exactly: hello from OpenAI",
            reasoning={"effort": "low"},   # remove this line if you switch to a non-reasoning model
            max_output_tokens=512,
            store=False,
        )
        LIVE["openai"] = (live_resp, (time.perf_counter() - t0) * 1000)
        print(live_resp.output_text, "|", live_resp.status, "|", live_resp.usage.input_tokens, "in /", live_resp.usage.output_tokens, "out |", live_resp._request_id)
    except openai.APIStatusError as e:
        print(f"OpenAI call failed: {type(e).__name__} {e.status_code}. See the README troubleshooting table.")
    except openai.APIConnectionError:
        print("OpenAI call failed: no network connection")
else:
    print("Skipped: set RUN_LIVE = True and put OPENAI_API_KEY in .env to run this cell.")

> **Observe:** on Claude, `output_tokens` is larger than the visible reply when the model thinks. On OpenAI, check `usage.output_tokens_details.reasoning_tokens`. Both are billed as output.

> **Try it:** set `max_tokens=5` on the Claude call. `stop_reason` becomes `max_tokens` and the text is cut off. Always check it before trusting the answer.

## 0.7 Counting tokens

Rule of thumb: about 4 characters per token for English. It is only a rough guide: code, other languages and each vendor's tokenizer change the number. Use each vendor's counting endpoint for real decisions (`messages.count_tokens` is free on Claude). [Theory](README.md#21-tokens) · [Theory](README.md#22-context-window)

In [ ]:
SAMPLES = {
    "English": "Please summarize the attached invoice and list any line items over $500.",
    "Python": "def total(items: list[dict]) -> float:\n    return sum(i['qty'] * i['price'] for i in items)",
    "German": "Bitte fassen Sie die beigefügte Rechnung zusammen und nennen Sie alle Posten über 500 $.",
    "Japanese": "添付の請求書を要約し、500ドルを超える項目をすべて挙げてください。",
}


def rough_tokens(text: str) -> int:
    return max(1, round(len(text) / 4))


counts = {name: {"chars": len(t), "rough": rough_tokens(t)} for name, t in SAMPLES.items()}
if RUN_LIVE and HAVE_CLAUDE_KEY:
    for name, t in SAMPLES.items():
        counts[name]["claude"] = anthropic.Anthropic().messages.count_tokens(
            model=CLAUDE_MODEL, messages=[{"role": "user", "content": t}]).input_tokens
if RUN_LIVE and HAVE_OPENAI_KEY:
    for name, t in SAMPLES.items():
        counts[name]["openai"] = openai.OpenAI().responses.input_tokens.count(model=OPENAI_MODEL, input=t).input_tokens

print(f"{'sample':<10} {'chars':>6} {'chars/4':>8} {'Claude':>7} {'OpenAI':>7}")
for name, c in counts.items():
    print(f"{name:<10} {c['chars']:>6} {c['rough']:>8} {c.get('claude', '-'):>7} {c.get('openai', '-'):>7}")
print("\nLive counts include a few tokens of message formatting." if RUN_LIVE else "\nSet RUN_LIVE = True to fill in the vendor columns.")

In [ ]:
# The API is stateless: every turn resends the whole history, so input grows each turn.
def history_growth(turns: int, tokens_per_turn: int, system_tokens: int) -> list[int]:
    return [system_tokens + tokens_per_turn * n for n in range(1, turns + 1)]


per_turn_input = history_growth(turns=20, tokens_per_turn=300, system_tokens=800)
print("input tokens on turns 1, 5, 10, 20:", [per_turn_input[i - 1] for i in (1, 5, 10, 20)])
print("total input tokens billed over 20 turns:", sum(per_turn_input))

> **Observe:** with `RUN_LIVE = True`, the non-English and code samples break the chars/4 rule, and Claude and OpenAI disagree on the same text. Never reuse one vendor's count for another, or one model generation's count for a newer one. In the history example, 20 turns of 300 tokens bill 79,000 input tokens, not 6,800.

> **Try it:** add a long system prompt (`system_tokens=5000`) and see how much of the total it becomes. That repeated prefix is what prompt caching discounts.

## 0.8 Estimating cost

Prices are USD per million tokens, from the official pricing pages on 2026-10-03 (standard tier, short context). Re-check them before estimating a real project. [Theory](README.md#step-7--put-cost-guardrails-in-place-before-the-first-call)

In [ ]:
PRICES = {  # USD per 1M tokens: input, output, cache read, cache write (Claude: 5-minute write)
    "claude-fable-5-1":          {"input": 10.00, "output": 50.00, "cache_read": 0.25, "cache_write": 12.50},
    "claude-opus-5-5":           {"input": 4.00,  "output": 20.00, "cache_read": 0.20, "cache_write": 5.00},
    "claude-sonnet-5-5":         {"input": 2.00,  "output": 10.00, "cache_read": 0.20, "cache_write": 2.50},
    "claude-haiku-4-5-20251001": {"input": 1.00,  "output": 5.00,  "cache_read": 0.10, "cache_write": 1.25},
    "gpt-6-astra":               {"input": 10.00, "output": 50.00, "cache_read": 1.00, "cache_write": 12.50},
    "gpt-6.1-sol":               {"input": 2.00,  "output": 10.00, "cache_read": 0.10, "cache_write": 2.50},
    "gpt-6-luna":                {"input": 0.10,  "output": 0.50,  "cache_read": 0.01, "cache_write": 0.125},
}
BATCH_DISCOUNT = 0.5   # both vendors' Batch APIs: 50% off


def price_for(model: str) -> dict:
    if model in PRICES:
        return PRICES[model]
    matches = [m for m in PRICES if m.startswith(model)]   # e.g. the alias "claude-haiku-4-5"
    if len(matches) == 1:
        return PRICES[matches[0]]
    raise KeyError(f"no price for {model!r}; add it from the pricing page")


def estimate_cost(model: str, input_tokens: int, output_tokens: int, *, cache_read: int = 0,
                  cache_write: int = 0, batch: bool = False) -> float:
    """USD for one request. `input_tokens` = uncached input only (cache reads/writes are separate)."""
    p = price_for(model)
    usd = (input_tokens * p["input"] + output_tokens * p["output"]
           + cache_read * p["cache_read"] + cache_write * p["cache_write"]) / 1e6
    return usd * (BATCH_DISCOUNT if batch else 1.0)


# The README worked example: 1,000 requests x 2,000 input x 500 output on Sonnet 5.5.
n = 1_000
base = n * estimate_cost("claude-sonnet-5-5", 2_000, 500)
batched = n * estimate_cost("claude-sonnet-5-5", 2_000, 500, batch=True)
cached = n * estimate_cost("claude-sonnet-5-5", 500, 500, cache_read=1_500) + estimate_cost("claude-sonnet-5-5", 0, 0, cache_write=1_500)
cached_input = cached - n * estimate_cost("claude-sonnet-5-5", 0, 500)
print(f"standard ${base:.2f} | batch ${batched:.2f} | 1,500-token cached prefix ${cached:.2f} "
      f"(input part ${cached_input:.2f} instead of $4.00)")
assert round(base, 2) == 9.00 and round(batched, 2) == 4.50

In [ ]:
# Same workload on every model: cost per request and per 1,000 requests.
print(f"{'model':<27} {'per request':>12} {'per 1,000':>10}")
for model in PRICES:
    one = estimate_cost(model, 2_000, 500)
    print(f"{model:<27} ${one:>11.5f} ${one * 1000:>9.2f}")

In [ ]:
def cost_from_usage(provider: str, model: str, usage) -> float:
    """Actual cost of one response from its `usage` block. The two vendors count cache tokens differently."""
    if provider == "anthropic":
        # Claude: input_tokens EXCLUDES cache reads and writes, which are reported separately.
        return estimate_cost(model, usage.input_tokens, usage.output_tokens,
                             cache_read=usage.cache_read_input_tokens or 0,
                             cache_write=usage.cache_creation_input_tokens or 0)
    # OpenAI: input_tokens INCLUDES cached tokens (and cache writes); reasoning is inside output_tokens.
    details = usage.input_tokens_details
    cached_ = details.cached_tokens or 0
    written = getattr(details, "cache_write_tokens", None) or 0
    return estimate_cost(model, usage.input_tokens - cached_ - written, usage.output_tokens,
                         cache_read=cached_, cache_write=written)


print(f"mock Claude reply: ${cost_from_usage('anthropic', 'claude-sonnet-5-5', mock_msg.usage):.6f}")
print(f"mock OpenAI reply: ${cost_from_usage('openai', 'gpt-6-luna', mock_resp.usage):.8f}")
for provider, (live_resp, _) in LIVE.items():
    model = CLAUDE_MODEL if provider == "claude" else OPENAI_MODEL
    print(f"live {provider}: ${cost_from_usage('anthropic' if provider == 'claude' else 'openai', model, live_resp.usage):.6f}")

> **Observe:** for the same 2,000-in / 500-out request, the most expensive model in the table costs 100x the cheapest. Model choice is the biggest cost lever, and it must be justified with evals, not guessed. Note also the vendor difference in `usage`: adding Claude's `cache_read_input_tokens` to `input_tokens` is correct; adding OpenAI's `cached_tokens` to `input_tokens` double-counts.

> **Try it:** estimate a month of a support bot: 30,000 conversations x 6 turns, each turn 3,000 input and 400 output tokens, on `claude-sonnet-5-5` and on `gpt-6-luna`. Then assume a 2,500-token cached prefix per turn.

## 0.9 Mini-project: a guarded smoke test with a cost log

Combine everything: **count → estimate the worst case → check a budget → call → log** one JSON line per call with the fields from the README observability section. It runs offline against the fake transport; with `RUN_LIVE = True` it runs the same pipeline against both real APIs. [Theory](README.md#34-observability) · [Theory](README.md#32-idempotency)

In [ ]:
BUDGET_PER_CALL_USD = 0.02


def smoke_test(provider: str, client, model: str, prompt: str, *, max_out: int = 1024) -> dict:
    # 1) Count before sending (free on Claude; on OpenAI it is one extra API request).
    if provider == "anthropic":
        counted = client.messages.count_tokens(model=model, messages=[{"role": "user", "content": prompt}]).input_tokens
    else:
        counted = client.responses.input_tokens.count(model=model, input=prompt).input_tokens
    # 2) Worst case: every allowed output token is generated (thinking/reasoning included).
    worst = estimate_cost(model, counted, max_out)
    if worst > BUDGET_PER_CALL_USD:
        raise RuntimeError(f"worst case ${worst:.4f} exceeds the ${BUDGET_PER_CALL_USD} budget")
    # 3) Call, timing it.
    t0 = time.perf_counter()
    if provider == "anthropic":
        r = client.messages.create(model=model, max_tokens=max_out,
                                   messages=[{"role": "user", "content": prompt}], **claude_effort_kwargs(model))
        text = "".join(b.text for b in r.content if b.type == "text")
        stop, cache_read = r.stop_reason, r.usage.cache_read_input_tokens or 0
    else:
        r = client.responses.create(model=model, input=prompt, reasoning={"effort": "low"},
                                    max_output_tokens=max_out, store=False)
        text, stop = r.output_text, r.status
        cache_read = r.usage.input_tokens_details.cached_tokens or 0
    # 4) One structured log line: what you need to debug, audit and bill.
    record = {
        "ts": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "provider": provider, "model": model, "request_id": r._request_id,
        "latency_ms": round((time.perf_counter() - t0) * 1000),
        "input_tokens": r.usage.input_tokens, "output_tokens": r.usage.output_tokens,
        "cache_read_input_tokens": cache_read, "stop_reason": stop,
        "estimated_cost_usd": round(cost_from_usage(provider, model, r.usage), 6),
        "worst_case_usd": round(worst, 6), "attempt": 1,
    }
    print(json.dumps(record))
    return {"text": text, "log": record}


mock_claude, mock_oai = mock_clients()
print(smoke_test("anthropic", mock_claude, "claude-sonnet-5-5", "Reply with exactly: hello from Claude")["text"])
print(smoke_test("openai", mock_oai, "gpt-6-luna", "Reply with exactly: hello from OpenAI")["text"])

In [ ]:
# Live version (opt-in). Uses your real keys and costs a fraction of a cent per vendor.
if RUN_LIVE:
    targets = [("anthropic", anthropic.Anthropic, CLAUDE_MODEL, HAVE_CLAUDE_KEY),
               ("openai", openai.OpenAI, OPENAI_MODEL, HAVE_OPENAI_KEY)]
    for provider, make_client, model, have_key in targets:
        if not have_key:
            print(f"{provider}: no key, skipped")
            continue
        try:
            smoke_test(provider, make_client(), model, f"Reply with exactly: hello from {provider}", max_out=512)
        except (anthropic.APIError, openai.APIError, RuntimeError) as e:
            print(f"{provider}: {type(e).__name__}: {e}")
else:
    print("Skipped: set RUN_LIVE = True to run the smoke test against the real APIs.")

> **Observe:** the budget check runs **before** the call, using the worst case (`max_tokens` fully used). That is the habit that prevents surprise bills: a runaway agent loop is just many calls that nobody priced.

> **Try it:** lower `BUDGET_PER_CALL_USD` to `0.005` and run the offline cell again. The Sonnet call is now refused before any money is spent (its worst case with `max_out=1024` is about $0.01), while the Luna call still passes.

## Recap

- Python 3.11+, the core packages and their lower bounds are checked from `requirements.txt`; extras are optional and per chapter.
- Keys are checked for presence and shape, never printed; `.env` must be git-ignored.
- Models generate one token at a time by sampling, so test properties and pass rates, not exact strings.
- Every call is an HTTPS `POST` with JSON. Read thinking-model output by block or item type, and read `stop_reason` / `status` before trusting it.
- The SDKs retry 408/409/429/5xx/529 for you; a spend-cap 429 is not transient.
- Count tokens with each vendor's endpoint, estimate the worst case, and log usage and cost for every call.

**Next:** [homework](02_homework.ipynb)